In [ ]:
from scrapling.fetchers import Fetcher

url = 'https://elcomercio.pe/deporte-total/futbol-mundial/video-espn-en-vivo-gratis-ver-futbol-libre-tv-partido-de-francia-vs-inglaterra-hoy-online-por-mundial-2026-via-disney-plus-noticia/'

page = Fetcher.get(url, stealthy_headers=True)

titulo = page.css('h1::text').get().strip()
titulo2 = page.css('h2::text').get().strip() 
fecha = page.css('time::attr(datetime)').get()
parrafos = [p.strip() for p in page.css('.sc__content p::text').getall() if p.strip()]
cuerpo = '\n\n'.join(parrafos)

caption = page.css('.story-contents__caption div::text').get()
if caption:
    caption = caption.strip()

print('Título:', titulo)
print('Fecha:', fecha)
print('Caption:', caption)
print('---')
print(cuerpo)


## Extracción masiva de artículos de fútbol

Estrategia: El Comercio arma cada sub-sección (`/deporte-total/futbol-mundial/`, `/deporte-total/futbol-peruano/`, etc.) con una lista inicial de artículos en el HTML, y un botón "cargar más" que trae el resto **vía JavaScript** (no funciona con `?page=2` en un simple GET — lo comprobé y devuelve exactamente los mismos links). Por eso, para juntar más artículos sin renderizar JS, conviene combinar varias sub-secciones relacionadas con fútbol.

⚠️ **Cuidado con el filtro de links**: un `a::attr(href)` sobre la página trae también artículos "recomendados" de otras secciones (horóscopo, feriados, elecciones, etc.) que también terminan en `-noticia/`. Por eso el filtro debe exigir que el link **empiece con la ruta de la sección** (`/deporte-total/...`), no solo que contenga la palabra "noticia".

Pasos:
1. Definir las sub-secciones de fútbol a recorrer.
2. Por cada sub-sección, sacar los links de artículos únicos.
3. Por cada link, entrar y extraer título / bajada / fecha / cuerpo (reutilizando los selectores que ya vimos).
4. Guardar todo en un `DataFrame` / CSV.
5. Poner una pausa (`time.sleep`) entre requests para no saturar el servidor.


In [3]:
import time
from scrapling.fetchers import Fetcher

SECCIONES_FUTBOL = [
    'https://elcomercio.pe/deporte-total/futbol-mundial/',
    'https://elcomercio.pe/deporte-total/futbol-peruano/',
    'https://elcomercio.pe/deporte-total/champions-league/',
    'https://elcomercio.pe/deporte-total/seleccion/',
]


def obtener_links_articulos(url_seccion):
    """Devuelve el set de URLs absolutas de artículos dentro de una sub-sección."""
    page = Fetcher.get(url_seccion, stealthy_headers=True)
    hrefs = page.css('a::attr(href)').getall()
    return {
        page.urljoin(h) for h in hrefs
        if h and h.startswith('/deporte-total/') and h.rstrip('/').endswith('noticia')
    }


def extraer_articulo(url):
    """Extrae título, bajada, fecha y cuerpo de una nota de El Comercio."""
    page = Fetcher.get(url, stealthy_headers=True)
    titulo = page.css('h1::text').get()
    bajada = page.css('h2::text').get()
    fecha = page.css('time::attr(datetime)').get()
    parrafos = [p.strip() for p in page.css('.sc__content p::text').getall() if p.strip()]
    return {
        'url': url,
        'titulo': titulo.strip() if titulo else None,
        'bajada': bajada.strip() if bajada else None,
        'fecha': fecha,
        'cuerpo': '\n\n'.join(parrafos),
    }


# 1) Recolectar links únicos de todas las sub-secciones de fútbol
todos_links = set()
for seccion in SECCIONES_FUTBOL:
    links = obtener_links_articulos(seccion)
    print(seccion, '->', len(links), 'links')
    todos_links.update(links)
    time.sleep(1)

print('Total de artículos únicos a extraer:', len(todos_links))


[2026-07-22 12:34:59] INFO: Fetched (200) <GET https://elcomercio.pe/deporte-total/futbol-mundial/> (referer: https://www.google.com/)


https://elcomercio.pe/deporte-total/futbol-mundial/ -> 46 links


[2026-07-22 12:35:01] INFO: Fetched (200) <GET https://elcomercio.pe/deporte-total/futbol-peruano/> (referer: https://www.google.com/)


https://elcomercio.pe/deporte-total/futbol-peruano/ -> 47 links


[2026-07-22 12:35:03] INFO: Fetched (200) <GET https://elcomercio.pe/deporte-total/champions-league/> (referer: https://www.google.com/)


https://elcomercio.pe/deporte-total/champions-league/ -> 57 links


[2026-07-22 12:35:05] INFO: Fetched (200) <GET https://elcomercio.pe/deporte-total/seleccion/> (referer: https://www.google.com/)


https://elcomercio.pe/deporte-total/seleccion/ -> 54 links
Total de artículos únicos a extraer: 204


In [4]:
# 2) Extraer cada artículo (con manejo de errores para que un fallo no corte todo el proceso)
import pandas as pd

resultados = []
for i, url in enumerate(sorted(todos_links), 1):
    try:
        data = extraer_articulo(url)
        resultados.append(data)
        print(f'[{i}/{len(todos_links)}] OK: {data["titulo"]}')
    except Exception as e:
        print(f'[{i}/{len(todos_links)}] ERROR en {url}: {e}')
    time.sleep(1)  # pausa entre requests para no saturar el servidor

df = pd.DataFrame(resultados)
df.to_csv('articulos_futbol.csv', index=False, encoding='utf-8-sig')
print('Guardado:', len(df), 'artículos en articulos_futbol.csv')
df.head()


[2026-07-22 12:35:29] INFO: Fetched (200) <GET https://elcomercio.pe/deporte-total/argentina/entre-lagrimas-lionel-scaloni-dejo-en-el-aire-su-continuidad-como-entrenador-de-la-seleccion-argentina-noticia/> (referer: https://www.google.com/)


[1/204] OK: ¡Entre lágrimas! Lionel Scaloni dejó en el aire su continuidad como entrenador de la selección argentina


[2026-07-22 12:35:31] INFO: Fetched (200) <GET https://elcomercio.pe/deporte-total/champions-league/a-que-hora-juega-psg-vs-bayern-munich-hoy-y-en-que-canales-ver-partido-por-semifinal-ida-de-champions-league-noticia/> (referer: https://www.google.com/)


[2/204] OK: ¿A qué hora jugaron PSG vs Bayern (5-4) por la semifinal ida de la Champions League?


[2026-07-22 12:35:34] INFO: Fetched (200) <GET https://elcomercio.pe/deporte-total/champions-league/arsenal-vs-atletico-madrid-en-vivo-online-por-semifinal-vuelta-de-champions-league-lbposting-noticia/> (referer: https://www.google.com/)


...

[2026-07-22 12:43:10] INFO: Fetched (200) <GET https://elcomercio.pe/deporte-total/seleccion/video-america-tv-en-vivo-gratis-donde-ver-peru-vs-espana-hoy-online-por-partido-amistoso-via-movistar-deportes-atv-america-tv-go-streaming-noticia/> (referer: https://www.google.com/)


[203/204] OK: VIDEO: ver goles del partido, España vs Perú (3-1) por amistoso FIFA 2026


[2026-07-22 12:43:13] INFO: Fetched (200) <GET https://elcomercio.pe/deporte-total/seleccion/video-america-tv-en-vivo-gratis-donde-ver-peru-vs-haiti-hoy-online-por-partido-amistoso-via-america-tv-go-canal-4-movistar-deportes-atv-noticia/> (referer: https://www.google.com/)


[204/204] OK: VIDEO: ver resumen y goles del partido, Perú vs Haití (2-1) por amistoso FIFA
Guardado: 204 artículos en articulos_futbol.csv


,url,titulo,bajada,fecha,cuerpo
0,https://elcomercio.pe/deporte-total/argentina/...,¡Entre lágrimas! Lionel Scaloni dejó en el air...,"Scaloni, que tiene contrato hasta diciembre, n...",2026-07-20T13:33:51.708Z,Lionel Scaloni dejó en el aire su continuidad ...
1,https://elcomercio.pe/deporte-total/champions-...,¿A qué hora jugaron PSG vs Bayern (5-4) por la...,Repasa los horarios del partido de PSG vs Baye...,2026-04-28T22:39:13.578Z,"A los 24′, Kvaratskhelia sacó provecho de un g..."
2,https://elcomercio.pe/deporte-total/champions-...,Arsenal venció 1-0 al Atlético Madrid y es el ...,"Con gol de Saka, Arsenal superó por la mínima ...",2026-05-05T22:57:42.905Z,Se jugarán cinco minutos de adición.\n\nBaena ...
3,https://elcomercio.pe/deporte-total/champions-...,Arsenal vs. Bayer Leverkusen (2-0): resumen y ...,Arsenal venció de local al Bayer Leverkusen en...,2026-03-17T22:07:22.911Z,"hizo la tarea en casa y, sin necesidad de una ..."
4,https://elcomercio.pe/deporte-total/champions-...,Arsenal vs. Bayer Leverkusen (1-1): resumen y ...,Arsenal rescató un empate en su visita al Baye...,2026-03-11T20:18:40.305Z,rescató un empate 1-1 ante\n\nen Alemania por ...
